In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

from langchain_chroma import Chroma

from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
import pandas as pd
books = pd.read_csv("books_cleaned.csv")

In [6]:
books["tagged_description"]

0       9780002005883: A NOVEL THAT READERS and critic...
1       9780002261982: A new 'Christie for Christmas' ...
2       9780006178736: A memorable, mesmerizing heroin...
3       9780006280897: Lewis' work on the nature of lo...
4       9780006280934: "In The Problem of Pain, C.S. L...
                              ...                        
5192    9788172235222: On A Train Journey Home To Nort...
5193    9788173031014: This book tells the tale of a m...
5194    9788179921623: Wisdom to Create a Life of Pass...
5195    9788185300535: This collection of the timeless...
5196    9789027712059: Since the three volume edition ...
Name: tagged_description, Length: 5197, dtype: object

In [8]:
books["tagged_description"].to_csv("tagged_description.txt",
                            sep="\n",
                            index=False,
                            header=False)

In [13]:
raw_documents = TextLoader("C:/Users/USER/PycharmProjects/Semantic_book_recommender/tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50, separator="\n")
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 1169, which is longer than the specified 500
Created a chunk of size 1215, which is longer than the specified 500
Created a chunk of size 961, which is longer than the specified 500
Created a chunk of size 844, which is longer than the specified 500
Created a chunk of size 882, which is longer than the specified 500
Created a chunk of size 1089, which is longer than the specified 500
Created a chunk of size 1190, which is longer than the specified 500
Created a chunk of size 514, which is longer than the specified 500
Created a chunk of size 753, which is longer than the specified 500
Created a chunk of size 729, which is longer than the specified 500
Created a chunk of size 722, which is longer than the specified 500
Created a chunk of size 1268, which is longer than the specified 500
Created a chunk of size 682, which is longer than the specified 500
Created a chunk of size 554, which is longer than the specified 500
Created a chunk of size 522, which is longe

In [14]:
documents[0]

Document(metadata={'source': 'C:/Users/USER/PycharmProjects/Semantic_book_recommender/tagged_description.txt'}, page_content='9780002005883: A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beaut

In [16]:
db_books = Chroma.from_documents(documents,
                                 embeddings)

In [17]:
query = "A book about psychology"
docs = db_books.similarity_search(query, k=10)
docs

[Document(id='b30516d3-736d-4bbf-b6dc-d16cddbab4a0', metadata={'source': 'C:/Users/USER/PycharmProjects/Semantic_book_recommender/tagged_description.txt'}, page_content='"9781593851170: ""Comprehensive and up to date, this tightly edited volume belongs on the desks of researchers and students in developmental psychology, comparative psychology, animal behavior, and evolutionary psychology, and will also be of interest to anthropologists. It is a richly informative text for advanced undergraduate- and graduate-level courses.""--BOOK JACKET."'),
 Document(id='f527e160-b000-47b7-a719-d56ff66eef63', metadata={'source': 'C:/Users/USER/PycharmProjects/Semantic_book_recommender/tagged_description.txt'}, page_content="9780345487421: A psychotherapist shows how to identify the fears that are inhibiting one's life, ranging from public speaking and intimacy to aging and rejection, and how to transform frustration and helplessness into power to create success in every aspect of life, in a twentiet

In [19]:
import re

isbn = re.findall(r"\d+", docs[0].page_content)[0]

books[books["isbn13"] == int(isbn)]

#books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
4898,9781593851170,1593851170,The Nature of Play,Anthony D. Pellegrini;Peter K. Smith,Psychology,http://books.google.com/books/content?id=Nukz6...,"""Comprehensive and up to date, this tightly ed...",2005.0,4.25,308.0,4.0,The Nature of Play: Great Apes and Humans,"9781593851170: ""Comprehensive and up to date, ..."


In [23]:
import re
import pandas as pd

def retrieve_semantic_recommendations(query: str, top_k: int = 10) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)

    books_list = []
    for r in recs:
        isbn = re.findall(r"\d+", r.page_content)[0]
        books_list.append(int(isbn))

    return books[books["isbn13"].isin(books_list)].head(top_k)

In [24]:
retrieve_semantic_recommendations("A book about neuroscience")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
94,9780060084387,0060084383,Modern Mind,Peter Watson,Science,http://books.google.com/books/content?id=JVdMS...,"From Freud to Babbitt, from Animal Farm to Sar...",2002.0,4.27,847.0,570.0,Modern Mind: An Intellectual History of the 20...,"9780060084387: From Freud to Babbitt, from Ani..."
101,9780060175641,0060175648,Identity,Milan Kundera,Fiction,http://books.google.com/books/content?id=D30Ex...,Milan Kundera's Identity translated from the F...,1998.0,3.68,176.0,260.0,Identity: A Novel,9780060175641: Milan Kundera's Identity transl...
113,9780060507404,0060507403,The Cheese Monkeys,Chip Kidd,Fiction,http://books.google.com/books/content?id=wkCm2...,"After 15 years of designing more than 1,500 bo...",2002.0,3.75,288.0,4570.0,The Cheese Monkeys: A Novel in Two Semesters,9780060507404: After 15 years of designing mor...
152,9780060570583,006057058X,The Perennial Philosophy,Aldous Huxley,Philosophy,http://books.google.com/books/content?id=YdaFE...,The Perennial Philosophy is defined by its aut...,2004.0,4.23,336.0,2959.0,The Perennial Philosophy,9780060570583: The Perennial Philosophy is def...
177,9780060595180,0060595183,The Doors of Perception and Heaven and Hell,Aldous Huxley,Philosophy,http://books.google.com/books/content?id=rz3NJ...,The acclaimed novelist and critic describes hi...,2004.0,3.94,187.0,27716.0,The Doors of Perception and Heaven and Hell,9780060595180: The acclaimed novelist and crit...
292,9780060925758,0060925752,Soul Mates,Thomas Moore,Psychology,http://books.google.com/books/content?id=7syEl...,This companion volume to Care of the Soul offe...,1994.0,4.00,288.0,4122.0,Soul Mates,9780060925758: This companion volume to Care o...
301,9780060930318,0060930314,Identity,Milan Kundera,Fiction,http://books.google.com/books/content?id=mXPU2...,There are situations in which we fail for a mo...,1999.0,3.68,168.0,13065.0,Identity: A Novel,9780060930318: There are situations in which w...
313,9780060936501,0060936509,The Best American Science Writing 2002,Matt Ridley;Alan Lightman,Science,http://books.google.com/books/content?id=p8yA_...,"If, as Matt Ridley suggests, science is simply...",2002.0,3.88,352.0,66.0,The Best American Science Writing 2002,"9780060936501: If, as Matt Ridley suggests, sc..."
404,9780064402453,0064402452,Racso and the Rats of NIMH,Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=MgoNv...,"‘Racso, a brash and boastful little rodent, is...",1988.0,3.76,288.0,3231.0,Racso and the Rats of NIMH,"9780064402453: ‘Racso, a brash and boastful li..."
600,9780140244915,0140244913,How the Mind Works,Steven Pinker,Philosophy,http://books.google.com/books/content?id=O521D...,"""Presented with extraordinary lucidity, cogenc...",1999.0,3.97,660.0,225.0,How the Mind Works,"9780140244915: ""Presented with extraordinary l..."
